<a href="https://colab.research.google.com/github/rushilnanga/acc-reality-check-project/blob/main/2026_11_18_%E2%80%94_ACC_Reality_Check_Starter_%E2%80%94_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · ACC Reality Check Starter

**Template — 2026-11-18 · Fall 2026**  

---

## ACC Reality Check: team notebook

Copy this notebook into your team repository and build the project in it. The sections follow the deliverables in the capstone PDF, so a grader can find each part where they expect it. Replace every TODO with your own work, and delete this paragraph when you are done.

Your published API lives in its own folder in the repository (for example `api/main.py`), because PythonAnywhere runs a Python file and not a notebook. Section 6 of this notebook only calls that API and draws a chart from what it returns.

### 0. Team, sources, and setup

**TODO:** list the team members, the two source URLs, and the date of each pull you used for the final results.

In [20]:
%pip install requests pandas

In [21]:
# TODO: imports and constants (URLs, the database file name, the raw data folder)
#Team: Rushil, Derek, Jackson, Luke
#Sources:
#Bart Torvik team ratings (CSV): https://barttorvik.com/2027_team_results.csv
#ESPN men's college basketball API (JSON)
# base URL: https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball
#scoreboard: `/scoreboard?groups=2&dates=YYYYMMDD&limit=200`
#game summary: `/summary?event=GAME_ID`

#Pulled: Torvik snapshots from 2026-11-18 on; ESPN games 2026-11-01 to 2026-11-17 (update at final run)

import os
import requests
import pandas as pd
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

#sources from pdf
TORVIK_URL = 'https://barttorvik.com/2027_team_results.csv'
SCOREBOARD_URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard'
SUMMARY_URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/summary'

DB_FILE = 'acc.db'
RAW_FOLDER = 'raw'
os.makedirs(RAW_FOLDER, exist_ok=True)

print('Raw files saved to :', RAW_FOLDER)


Raw files saved to : raw


### 1. Extract

**TODO:** pull the Torvik CSV and the ESPN scoreboard and summary JSON. Save every raw response to disk, and read from the saved copy when it exists so that a rerun does not call the source again.

In [22]:
# TODO: Torvik ratings snapshot

#get current date time
import shutil

today = datetime.now(ZoneInfo("America/New_York")).strftime("%Y-%m-%d")

os.makedirs(RAW_FOLDER, exist_ok=True)

torvik_file = os.path.join(RAW_FOLDER, f"torvik_{today}.csv")

shutil.copy("2027_team_results.csv", torvik_file)

torvik_df = pd.read_csv(torvik_file)

torvik_df.head()

,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Houston,B12,6-0,124.668627,3,93.868441,3,0.963148,1,...,0.0,0.0,0,0,0.830566,0,2.489538,1,26,67.597053
1,2,Illinois,B10,6-1,127.017731,1,97.161096,21,0.956123,2,...,0.0,0.0,0,0,0.836518,0,1.018647,20,125,74.318835
2,3,Florida,SEC,6-1,124.838540,2,96.375247,13,0.951472,3,...,0.0,0.0,0,0,0.858313,0,2.282661,2,75,68.531416
3,4,Kansas,B12,6-1,122.374076,6,94.520893,4,0.951201,4,...,0.0,0.0,0,0,0.828915,0,1.351684,7,151,72.861257
4,5,Duke,ACC,4-2,119.918447,20,93.305187,1,0.947138,5,...,0.0,0.0,0,0,0.804717,0,-0.311248,84,338,71.526314


In [23]:
# TODO: ESPN scoreboard, one request per date
import json

start = date(2026, 11, 1)
end = datetime.now(ZoneInfo('America/New_York')).date() - pd.Timedelta(days=1)

day = start
while day <= end:
    date_str = day.strftime('%Y%m%d')
    scoreboard_file = os.path.join(RAW_FOLDER, f'scoreboard_{date_str}.json')
    if not os.path.exists(scoreboard_file):
        response = requests.get(SCOREBOARD_URL,
                                params={'groups' : 2, 'dates': date_str, 'limit': 200},
                                headers={'User-Agent': 'Mozilla/5.0'}, timeout = 30)
        response.raise_for_status()
        open(scoreboard_file, 'w').write(response.text)
    day += pd.Timedelta(days=1)

print(len(os.listdir(RAW_FOLDER)), 'raw files saved to :', RAW_FOLDER)

134 raw files saved to : raw


In [24]:
# TODO: ESPN game summaries, one request per finished game

for sb in os.listdir(RAW_FOLDER):
    if sb.startswith('scoreboard_') and sb.endswith('.json'):
        scoreboard_file = os.path.join(RAW_FOLDER, sb)
        with open(scoreboard_file) as f:
            scoreboard_data = json.load(f)
        for event in scoreboard_data['events']:
            if event['competitions'][0]['status']['type']['completed']:
                game_id = event['id']
                summary_file = os.path.join(RAW_FOLDER, f'summary_{game_id}.json')
                if not os.path.exists(summary_file):
                    response = requests.get(SUMMARY_URL,
                                            params={'event': game_id},
                                            headers={'User-Agent': 'Mozilla/5.0'}, timeout = 30)
                    response.raise_for_status()
                    open(summary_file, 'w').write(response.text)

print(len(os.listdir(RAW_FOLDER)), 'raw files saved to :', RAW_FOLDER)


134 raw files saved to : raw


In [25]:
#Check
files = os.listdir(RAW_FOLDER)
torvik_files = [f for f in files if f.startswith('torvik_')]
scoreboard_files = [f for f in files if f.startswith('scoreboard_')]
summary_files = [f for f in files if f.startswith('summary_')]

print(f'Torvik files: {len(torvik_files)}')
print(f'Scoreboard files: {len(scoreboard_files)}')
print(f'Summary files: {len(summary_files)}')
print("ACC teams in Torvik ratings snapshot:", (torvik_df['conf'] == 'ACC').sum())

Torvik files: 1
Scoreboard files: 23
Summary files: 110
ACC teams in Torvik ratings snapshot: 18


### 2. Transform

**TODO:** one function or clearly labeled cell per step, each printing a row count.

In [45]:
# TODO: the team name crosswalk, with the unmatched names printed before and after
import re
import json
import sqlite3
from pathlib import Path

DECISIONS = []

def log(step, decision, rows):
    DECISIONS.append({
        'step': step,
        'decision': decision,
        'rows_affected': rows
    })

def normalize(name):
    name = str(name).lower()
    name = re.sub(r'\bst\.?\b', 'state', name)
    return re.sub(r'[^a-z0-9]', '', name)

raw = Path(RAW_FOLDER)

ratings_list = []

for path in sorted(raw.glob('torvik_*.csv')):
    frame = pd.read_csv(path)
    frame['snapshot_date'] = path.stem.replace('torvik_', '')
    ratings_list.append(frame)

ratings_raw = pd.concat(ratings_list, ignore_index=True)

latest = ratings_raw[
    ratings_raw['snapshot_date'] == ratings_raw['snapshot_date'].max()
].copy()

acc_names = set(latest.loc[latest['conf'] == 'ACC', 'team'])

torvik_lookup = {
    normalize(name): name
    for name in latest['team']
}

events = {}

for path in sorted(raw.glob('scoreboard_*.json')):
    data = json.loads(path.read_text())

    for event in data['events']:
        events[str(event['id'])] = event

espn_teams = {}

for event in events.values():
    for competitor in event['competitions'][0]['competitors']:
        team = competitor['team']
        espn_teams[str(team['id'])] = team['location']

teams = pd.DataFrame([
    {'team_id': team_id, 'espn_name': name}
    for team_id, name in espn_teams.items()
])

teams['torvik_name'] = teams['espn_name'].map(
    lambda name: torvik_lookup.get(normalize(name))
)

before = teams['torvik_name'].isna()

print('Unmatched before manual fixes:', before.sum())
print(teams.loc[before, 'espn_name'].tolist())

NAME_MAP = {
    'NC State': 'N.C. State',
    'Miami': 'Miami FL',
    'Miami (FL)': 'Miami FL',
    'Pitt': 'Pittsburgh',
    'California': 'California',
    'Cal': 'California',
    'Florida State': 'Florida St.',
    'North Carolina State': 'N.C. State',
    'Southern Methodist': 'SMU',
    'UConn': 'Connecticut',
    'UMass': 'Massachusetts',
    'Ole Miss': 'Mississippi',
    'LSU': 'LSU',
    'UCF': 'UCF',
    'BYU': 'BYU',
    'UNLV': 'UNLV',
    'VCU': 'VCU',
    "Saint Mary's": "Saint Mary's",
    'American University': 'American',
    'Texas A&M-Corpus Christi': 'Texas A&M Corpus Chris',
    'Loyola Maryland': 'Loyola MD',
}

for i, row in teams[before].iterrows():
    corrected = NAME_MAP.get(row['espn_name'])

    if corrected in set(latest['team']):
        teams.loc[i, 'torvik_name'] = corrected

after = teams['torvik_name'].isna()

teams['is_acc'] = teams['torvik_name'].isin(acc_names).astype(int)

print('\nUnmatched after manual fixes:', after.sum())
print(teams.loc[after, 'espn_name'].tolist())

print('\nTotal ESPN teams:', len(teams))
print('ACC teams matched:', teams['is_acc'].sum())
print('Missing ACC teams:', acc_names - set(teams['torvik_name']))

assert not teams['team_id'].duplicated().any()
assert not teams['torvik_name'].dropna().duplicated().any()
assert teams['is_acc'].sum() == 18

log('Crosswalk', 'Matched team names automatically', int((~before).sum()))
log('Crosswalk', 'Matched team names manually', int(before.sum() - after.sum()))
log('Crosswalk', 'Retained unmatched teams without ratings', int(after.sum()))

team_crosswalk = teams[
    ['team_id', 'espn_name', 'torvik_name']
].copy()

teams.head()

Unmatched before manual fixes: 6
['Miami', 'Adams State', 'American University', 'Texas A&M-Corpus Christi', 'Loyola Maryland', 'Ole Miss']

Unmatched after manual fixes: 1
['Adams State']

Total ESPN teams: 116
ACC teams matched: 18
Missing ACC teams: set()


,team_id,espn_name,torvik_name,is_acc
0,222,Villanova,Villanova,0
1,87,Notre Dame,Notre Dame,1
2,57,Florida,Florida,0
3,2390,Miami,Miami FL,1
4,150,Duke,Duke,1


In [28]:
# TODO: games, with the Eastern game date derived from the UTC start time
game_rows = []
unfinished = 0
date_changes = 0

for game_id, event in events.items():

    comp = event['competitions'][0]

    if not comp['status']['type']['completed']:
        unfinished += 1
        continue

    competitors = comp['competitors']

    home = next(c for c in competitors if c['homeAway'] == 'home')
    away = next(c for c in competitors if c['homeAway'] == 'away')

    utc = pd.to_datetime(comp['date'], utc=True)
    eastern = utc.tz_convert('America/New_York')

    if utc.date() != eastern.date():
        date_changes += 1

    game_rows.append({
        'game_id': game_id,
        'game_date': eastern.date().isoformat(),
        'start_utc': utc.isoformat(),
        'home_team_id': str(home['team']['id']),
        'away_team_id': str(away['team']['id']),
        'home_score': int(home['score']),
        'away_score': int(away['score']),
        'neutral_site': int(comp.get('neutralSite', False))
    })

games = pd.DataFrame(game_rows)

assert games['game_id'].is_unique

print('Completed games:', len(games))
print('Unfinished games excluded:', unfinished)
print('Dates changed after timezone conversion:', date_changes)
print('Neutral-site games:', games['neutral_site'].sum())

log('Games', 'Excluded unfinished games', unfinished)
log('Games', 'Converted UTC timestamps to Eastern time', len(games))
log('Games', 'Corrected calendar dates after conversion', date_changes)

games.head()

Completed games: 110
Unfinished games excluded: 0
Dates changed after timezone conversion: 79
Neutral-site games: 19


,game_id,game_date,start_utc,home_team_id,away_team_id,home_score,away_score,neutral_site
0,401920982,2026-11-01,2026-11-01T14:30:00+00:00,222,87,72,74,1
1,401906769,2026-11-02,2026-11-02T23:00:00+00:00,57,2390,77,67,1
2,401909562,2026-11-02,2026-11-02T23:30:00+00:00,150,349,84,50,0
3,401918947,2026-11-02,2026-11-03T02:00:00+00:00,97,236,84,70,0
4,401918073,2026-11-02,2026-11-03T00:00:00+00:00,228,56,85,72,0


In [29]:
# TODO: team_games, one row per team per game, with margin and possessions

summaries = {}

for path in raw.glob('summary_*.json'):
    game_id = path.stem.replace('summary_', '')
    summaries[game_id] = json.loads(path.read_text())

def numeric(value):
    return pd.to_numeric(value, errors='coerce')

def attempted(value):
    parts = str(value).split('-')

    if len(parts) == 2:
        return numeric(parts[1])

    return float('nan')

def made(value):
    parts = str(value).split('-')

    if len(parts) == 2:
        return numeric(parts[0])

    return float('nan')

team_game_rows = []

for game in games.to_dict('records'):

    summary = summaries.get(game['game_id'], {})
    boxscore = summary.get('boxscore', {})
    box_teams = boxscore.get('teams', [])

    for side in ['home', 'away']:

        other = 'away' if side == 'home' else 'home'

        team_id = game[f'{side}_team_id']
        opponent_id = game[f'{other}_team_id']

        points = game[f'{side}_score']
        against = game[f'{other}_score']

        stats = {}

        for entry in box_teams:
            if str(entry['team']['id']) == team_id:
                stats = {
                    s['name']: s['displayValue']
                    for s in entry.get('statistics', [])
                }
                break

        fg = stats.get('fieldGoalsMade-fieldGoalsAttempted')
        ft = stats.get('freeThrowsMade-freeThrowsAttempted')
        three = stats.get('threePointFieldGoalsMade-threePointFieldGoalsAttempted')

        fgm = made(fg)
        fga = attempted(fg)
        ftm = made(ft)
        fta = attempted(ft)

        oreb = numeric(stats.get('offensiveRebounds'))
        turnovers = numeric(stats.get('totalTurnovers', stats.get('turnovers')))

        possessions = fga - oreb + turnovers + 0.475 * fta

        team_game_rows.append({
            'game_id': game['game_id'],
            'team_id': team_id,
            'opponent_id': opponent_id,
            'game_date': game['game_date'],
            'location': 'neutral' if game['neutral_site'] else side,
            'points_for': points,
            'points_against': against,
            'margin': points - against,
            'win': int(points > against),
            'fgm': fgm,
            'fga': fga,
            'ftm': ftm,
            'fta': fta,
            'three_pm': made(three),
            'three_pa': attempted(three),
            'offensive_rebounds': oreb,
            'defensive_rebounds': numeric(stats.get('defensiveRebounds')),
            'rebounds': numeric(stats.get('totalRebounds')),
            'assists': numeric(stats.get('assists')),
            'steals': numeric(stats.get('steals')),
            'blocks': numeric(stats.get('blocks')),
            'turnovers': turnovers,
            'possessions': possessions
        })

team_games = pd.DataFrame(team_game_rows)

assert len(team_games) == 2 * len(games)
assert not team_games.duplicated(['game_id', 'team_id']).any()

print('Completed games:', len(games))
print('Team-game rows:', len(team_games))
print('Possession estimates:', team_games['possessions'].notna().sum())
print('Missing possession estimates:', team_games['possessions'].isna().sum())

log('Team games', 'Reshaped games into two team perspectives', len(team_games))
log('Team games', 'Calculated scoring margins', len(team_games))
log('Team games', 'Calculated estimated possessions',
    int(team_games['possessions'].notna().sum()))
log('Team games', 'Retained missing possessions as null',
    int(team_games['possessions'].isna().sum()))

team_games.head()

Completed games: 110
Team-game rows: 220
Possession estimates: 220
Missing possession estimates: 0


,game_id,team_id,opponent_id,game_date,location,points_for,points_against,margin,win,fgm,...,three_pm,three_pa,offensive_rebounds,defensive_rebounds,rebounds,assists,steals,blocks,turnovers,possessions
0,401920982,222,87,2026-11-01,neutral,72,74,-2,0,24,...,11,31,10,27,37,12,6,4,20,70.550
1,401920982,87,222,2026-11-01,neutral,74,72,2,1,23,...,9,24,17,20,37,14,14,2,11,70.875
2,401906769,57,2390,2026-11-02,neutral,77,67,10,1,23,...,11,26,12,32,44,17,7,5,12,69.625
3,401906769,2390,57,2026-11-02,neutral,67,77,-10,0,21,...,4,17,13,26,39,6,5,0,11,72.250
4,401909562,150,349,2026-11-02,home,84,50,34,1,29,...,4,22,14,23,37,14,16,6,13,70.350


In [30]:
# TODO: player_games
player_rows = []
dnp_count = 0

valid_games = set(games['game_id'])

def get_minutes(value):
    if ':' in str(value):
        minutes, seconds = str(value).split(':')
        return numeric(minutes) + numeric(seconds) / 60

    return numeric(value)

for game_id, summary in summaries.items():

    if game_id not in valid_games:
        continue

    for team in summary.get('boxscore', {}).get('players', []):

        team_id = str(team['team']['id'])

        for group in team.get('statistics', []):

            keys = group.get('keys', [])
            labels = group.get('labels', [])

            for player in group.get('athletes', []):

                values = player.get('stats', [])

                if player.get('didNotPlay', False) or not values:
                    dnp_count += 1
                    continue

                stats = dict(zip(keys, values))
                stats.update(dict(zip(labels, values)))

                athlete = player['athlete']

                player_rows.append({
                    'game_id': game_id,
                    'team_id': team_id,
                    'player_id': str(athlete['id']),
                    'player_name': athlete['displayName'],
                    'minutes': get_minutes(stats.get('minutes', stats.get('MIN'))),
                    'points': numeric(stats.get('points', stats.get('PTS'))),
                    'rebounds': numeric(stats.get('rebounds', stats.get('REB'))),
                    'assists': numeric(stats.get('assists', stats.get('AST'))),
                    'steals': numeric(stats.get('steals', stats.get('STL'))),
                    'blocks': numeric(stats.get('blocks', stats.get('BLK'))),
                    'turnovers': numeric(stats.get('turnovers', stats.get('TO')))
                })

player_games = pd.DataFrame(player_rows)

before_count = len(player_games)

player_games = player_games.drop_duplicates(
    subset=['game_id', 'player_id']
)

duplicates_removed = before_count - len(player_games)

print('Player-game rows:', len(player_games))
print('Unique players:', player_games['player_id'].nunique())
print('Games with player statistics:', player_games['game_id'].nunique())
print('Players who did not play:', dnp_count)
print('Duplicates removed:', duplicates_removed)
print('Missing points:', player_games['points'].isna().sum())

log('Player games', 'Flattened player box scores', len(player_games))
log('Player games', 'Excluded players who did not play', dnp_count)
log('Player games', 'Removed duplicate player-game records', duplicates_removed)

player_games.head()

Player-game rows: 2487
Unique players: 1376
Games with player statistics: 110
Players who did not play: 917
Duplicates removed: 0
Missing points: 0


,game_id,team_id,player_id,player_name,minutes,points,rebounds,assists,steals,blocks,turnovers
0,401929459,6,5313205,Peyton Law,17,4,1,1,1,0,1
1,401929459,6,5313202,Stephen Williams,28,7,7,2,1,0,1
2,401929459,6,5461609,Diego Roman,18,3,2,1,0,0,2
3,401929459,6,5461606,Jalen Langsy,33,22,2,2,0,0,2
4,401929459,6,5313201,Samuel Shoptaw,24,2,1,4,1,0,2


### 3. Load into SQLite

**TODO:** create the tables with primary keys, load them so that a rerun cannot duplicate rows, and print the row count of every table.

In [31]:
# TODO: schema
conn = sqlite3.connect(DB_FILE)

conn.executescript("""
CREATE TABLE IF NOT EXISTS teams (
    team_id TEXT PRIMARY KEY,
    espn_name TEXT,
    torvik_name TEXT,
    is_acc INTEGER
);

CREATE TABLE IF NOT EXISTS team_crosswalk (
    team_id TEXT PRIMARY KEY,
    espn_name TEXT,
    torvik_name TEXT
);

CREATE TABLE IF NOT EXISTS ratings (
    torvik_name TEXT,
    snapshot_date TEXT,
    conf TEXT,
    record TEXT,
    wins INTEGER,
    losses INTEGER,
    adjoe REAL,
    adjde REAL,
    barthag REAL,
    rank INTEGER,
    adjt REAL,
    sos REAL,
    wab REAL,
    PRIMARY KEY (torvik_name, snapshot_date)
);

CREATE TABLE IF NOT EXISTS games (
    game_id TEXT PRIMARY KEY,
    game_date TEXT,
    start_utc TEXT,
    home_team_id TEXT,
    away_team_id TEXT,
    home_score INTEGER,
    away_score INTEGER,
    neutral_site INTEGER
);

CREATE TABLE IF NOT EXISTS team_games (
    game_id TEXT,
    team_id TEXT,
    opponent_id TEXT,
    game_date TEXT,
    location TEXT,
    points_for INTEGER,
    points_against INTEGER,
    margin INTEGER,
    win INTEGER,
    fgm REAL,
    fga REAL,
    ftm REAL,
    fta REAL,
    three_pm REAL,
    three_pa REAL,
    offensive_rebounds REAL,
    defensive_rebounds REAL,
    rebounds REAL,
    assists REAL,
    steals REAL,
    blocks REAL,
    turnovers REAL,
    possessions REAL,
    PRIMARY KEY (game_id, team_id)
);

CREATE TABLE IF NOT EXISTS player_games (
    game_id TEXT,
    team_id TEXT,
    player_id TEXT,
    player_name TEXT,
    minutes REAL,
    points REAL,
    rebounds REAL,
    assists REAL,
    steals REAL,
    blocks REAL,
    turnovers REAL,
    PRIMARY KEY (game_id, player_id)
);
""")

conn.commit()

print('Database created:', DB_FILE)

pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table'",
    conn
)

Database created: acc.db


,name
0,teams
1,team_crosswalk
2,ratings
3,games
4,team_games
5,player_games


In [32]:
# TODO: load, then prove that a second run leaves the counts unchanged
ratings = ratings_raw[
    ['team', 'snapshot_date', 'conf', 'record',
     'adjoe', 'adjde', 'barthag', 'rank.1',
     'adjt', 'sos', 'WAB']
].copy()

ratings = ratings.rename(columns={
    'team': 'torvik_name',
    'rank.1': 'rank',
    'WAB': 'wab'
})

records = ratings['record'].str.extract(r'(\d+)-(\d+)')

ratings['wins'] = pd.to_numeric(records[0])
ratings['losses'] = pd.to_numeric(records[1])

def load_table(table, frame):

    columns = ', '.join(f'"{c}"' for c in frame.columns)
    placeholders = ', '.join(['?'] * len(frame.columns))

    query = f"""
    INSERT OR REPLACE INTO {table}
    ({columns})
    VALUES ({placeholders})
    """

    clean = frame.astype(object).where(pd.notna(frame), None)

    conn.executemany(
        query,
        list(clean.itertuples(index=False, name=None))
    )

def load_database():

    with conn:
        load_table('teams', teams)
        load_table('team_crosswalk', team_crosswalk)
        load_table('ratings', ratings)
        load_table('games', games)
        load_table('team_games', team_games)
        load_table('player_games', player_games)

tables = [
    'teams',
    'team_crosswalk',
    'ratings',
    'games',
    'team_games',
    'player_games'
]

def get_counts():

    return {
        table: conn.execute(
            f'SELECT COUNT(*) FROM {table}'
        ).fetchone()[0]
        for table in tables
    }

load_database()
first_load = get_counts()

load_database()
second_load = get_counts()

comparison = pd.DataFrame({
    'First load': first_load,
    'Second load': second_load
})

comparison['Unchanged'] = (
    comparison['First load'] == comparison['Second load']
)

display(comparison)

assert first_load == second_load

print('PASS: Running the load twice does not create duplicates.')

,First load,Second load,Unchanged
teams,116,116,True
team_crosswalk,116,116,True
ratings,365,365,True
games,110,110,True
team_games,220,220,True
player_games,2487,2487,True


PASS: Running the load twice does not create duplicates.


In [33]:
acc_count = conn.execute("""
SELECT COUNT(*)
FROM teams
WHERE is_acc = 1
""").fetchone()[0]

game_count = conn.execute("""
SELECT COUNT(*)
FROM games
""").fetchone()[0]

team_game_count = conn.execute("""
SELECT COUNT(*)
FROM team_games
""").fetchone()[0]

invalid_games = conn.execute("""
SELECT COUNT(*)
FROM (
    SELECT game_id
    FROM team_games
    GROUP BY game_id
    HAVING COUNT(*) != 2
)
""").fetchone()[0]

missing_possessions = conn.execute("""
SELECT COUNT(*)
FROM team_games
WHERE possessions IS NULL
""").fetchone()[0]

missing_player_points = conn.execute("""
SELECT COUNT(*)
FROM player_games
WHERE points IS NULL
""").fetchone()[0]

print('ACC teams:', acc_count)
print('Completed games:', game_count)
print('Team-game rows:', team_game_count)
print('Games without exactly two team rows:', invalid_games)
print('Missing possessions:', missing_possessions)
print('Missing player points:', missing_player_points)

assert acc_count == 18
assert team_game_count == 2 * game_count
assert invalid_games == 0

print('\nPASS: Team and game validation successful.')

display(pd.read_sql_query("""
SELECT snapshot_date, COUNT(*) AS teams
FROM ratings
GROUP BY snapshot_date
ORDER BY snapshot_date
""", conn))

display(pd.read_sql_query("""
SELECT
    t.torvik_name,
    COUNT(*) AS games,
    SUM(tg.win) AS wins,
    SUM(1 - tg.win) AS losses
FROM team_games tg
JOIN teams t ON tg.team_id = t.team_id
WHERE t.is_acc = 1
GROUP BY t.team_id
ORDER BY wins DESC
""", conn))

ACC teams: 18
Completed games: 110
Team-game rows: 220
Games without exactly two team rows: 0
Missing possessions: 0
Missing player points: 0

PASS: Team and game validation successful.


,snapshot_date,teams
0,2026-11-24,365


,torvik_name,games,wins,losses
0,Virginia,7,7,0
1,Notre Dame,7,6,1
2,Virginia Tech,7,6,1
3,SMU,6,6,0
4,Pittsburgh,6,6,0
5,Boston College,6,6,0
6,Georgia Tech,6,5,1
7,Florida St.,6,5,1
8,California,7,5,2
9,Stanford,6,5,1


### 4. Questions

Each question gets a SQL query run against your database, a chart if the PDF asks for one, and an answer written in full sentences that includes the number it rests on.

#### Q1. Record versus rating

In [34]:
# TODO: SQL

In [35]:
# TODO: chart 1

_Q1 answer._

#### Q2. Strength of the opponents so far

In [36]:
# TODO: SQL

In [37]:
# TODO: chart 2

_Q2 answer._

#### Q3. Close games and recent form

In [38]:
# TODO: SQL, including a window function for the rolling margin

In [39]:
# TODO: chart 3

_Q3 answer._

#### Q4. Pace and points per possession

In [40]:
# TODO: SQL

_Q4 answer._

#### Q5. Who carries each team

In [41]:
# TODO: SQL

_Q5 answer._

#### Q6. Does the rating predict the winner?

In [42]:
# TODO: SQL that uses the latest snapshot taken before each game

_Q6 answer, with the number of games it covers._

### 5. Decision log

**TODO:** every decision that changed the data, with the number of rows it affected.

In [43]:
# TODO
decision_log = pd.DataFrame(DECISIONS)

display(decision_log)

,step,decision,rows_affected
0,Crosswalk,Matched team names automatically,110
1,Crosswalk,Matched team names manually,5
2,Crosswalk,Retained unmatched teams without ratings,1
3,Games,Excluded unfinished games,0
4,Games,Converted UTC timestamps to Eastern time,110
5,Games,Corrected calendar dates after conversion,79
6,Team games,Reshaped games into two team perspectives,220
7,Team games,Calculated scoring margins,220
8,Team games,Calculated estimated possessions,220
9,Team games,Retained missing possessions as null,0


### 6. The published API

**TODO:** the live address, the list of routes, and chart 4, drawn from JSON returned by your own API.

In [44]:
API_URL = 'TODO'

# TODO: call your routes with requests and draw chart 4

### 7. Reflection

**TODO:** answer the five reflection questions from the PDF, with every member contributing.

_reflection here_

### 8. Presentation video

**TODO:** the link to your team's recorded presentation, which runs no more than eight minutes and includes every member. Put the same link in the README.

_video link here_